# Credentials & panzer

BinPan delegates **all** secret management to [`panzer`](https://pypi.org/project/panzer/)'s `CredentialManager`. Credentials live encrypted in `~/.panzer_creds`, are bound to the machine that created them, and are requested interactively the first time they are needed.

**You only need API keys for account/signed endpoints** (wallet balances, fees, account info). Market data — klines, trades, order book, exchange info — is **public and needs no keys**.

> BinPan has **no order, no withdraw** methods. For peace of mind, create a Binance API key with trading disabled (read-only).

In [ ]:
import binpan

## Public data needs no keys

Candles, trades, depth and exchange info work out of the box:

In [ ]:
sym = binpan.Symbol(symbol='BTCUSDT', tick_interval='1h', limit=10)
sym.df[['Open', 'High', 'Low', 'Close', 'Volume']].tail(3)

## Where credentials live

Everything is stored in `~/.panzer_creds` (sensitive values encrypted). BinPan's thin wrapper lives in `binpan.core.secrets`:

In [ ]:
import os
print('Credentials file:', os.path.expanduser('~/.panzer_creds'))
# the wrapper around panzer's CredentialManager
from binpan.core.secrets import get_secret, set_secret, get_json_secret, set_json_secret
print('helpers:', [get_secret.__name__, set_secret.__name__, get_json_secret.__name__, set_json_secret.__name__])

## Adding your Binance API keys

Two ways:

**1) Automatic prompt.** The first time you call a signed method (e.g. a `Wallet`/`Exchange` account endpoint), panzer asks for `api_key` and `api_secret` (hidden input) and stores them encrypted. Nothing to do up front.

**2) Programmatically** (run once, then delete the cell). The names contain `api_key`/`secret`, so panzer encrypts them automatically:

```python
from binpan.core.secrets import set_secret
set_secret('api_key', 'YOUR_BINANCE_API_KEY')
set_secret('api_secret', 'YOUR_BINANCE_API_SECRET')
```

From then on, signed calls just work. Read a value back (decrypted) with `get_secret('api_key')` — only do this on the machine where you stored it.

## Other credentials (databases, etc.)

The same mechanism handles any secret. Sensitive names (containing `secret`, `api_key`, `password` or `_id`) are encrypted; the rest are stored as plain text. Config dicts (Redis/Sentinel) use the JSON helpers:

```python
from binpan.core.secrets import set_secret, set_json_secret
set_secret('postgresql_password', '...')          # encrypted (contains 'password')
set_secret('postgresql_host', '192.168.1.10')      # plain text
set_json_secret('redis_conf', {'host': '127.0.0.1', 'port': 6379})
```

## Security notes

- Values are encrypted with AES and the key is **derived from your user + CPU**, so `~/.panzer_creds` only decrypts on the same machine.
- BinPan never logs secrets in plain text and exposes **no order or withdraw** methods.
- To reset a credential, edit or delete the line in `~/.panzer_creds` (or the whole file) and it will be requested again on next use.

## Thank you

See the other notebooks for analysis, plotting and exchange features.